# Bayesian Change Point Analysis - Brent Oil Prices

This notebook implements Bayesian change point detection to identify structural breaks in Brent oil prices and correlate them with historical events.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully")

## 1. Data Loading and Preparation

In [ ]:
# Load Brent oil price data
df = pd.read_csv('../data/BrentOilPrices.csv')
df['date'] = pd.to_datetime(df['Date'])
df['price'] = df['Price']
df.set_index('date', inplace=True)
df.sort_index(inplace=True)

# Load events data
events_df = pd.read_csv('../data/key_events_oil_prices.csv')
events_df['date'] = pd.to_datetime(events_df['date'])

print(f"Dataset: {len(df)} observations from {df.index.min()} to {df.index.max()}")
print(f"Events: {len(events_df)} major events")
print(f"\nPrice Statistics:")
print(df['price'].describe())

## 2. Bayesian Change Point Model

We implement a single change point model to detect a structural break in the mean price level.

In [ ]:
# Prepare data for modeling
prices = df['price'].values
n = len(prices)
time_indices = np.arange(n)

print(f"Building model with {n} data points...")

# Define the Bayesian change point model
with pm.Model() as change_point_model:
    # Prior for change point (uniform over middle 50% of data)
    tau = pm.DiscreteUniform('tau', lower=int(0.25*n), upper=int(0.75*n))
    
    # Priors for before and after means
    mu_before = pm.Normal('mu_before', mu=50, sigma=20)
    mu_after = pm.Normal('mu_after', mu=50, sigma=20)
    
    # Prior for standard deviation
    sigma = pm.HalfNormal('sigma', sigma=10)
    
    # Switch function to select appropriate mean
    mu = pm.math.switch(time_indices < tau, mu_before, mu_after)
    
    # Likelihood
    likelihood = pm.Normal('likelihood', mu=mu, sigma=sigma, observed=prices)
    
    # Sample using MCMC
    print("Running MCMC sampling...")
    trace = pm.sample(2000, tune=1000, chains=2, target_accept=0.9, progressbar=False, cores=1)

print("Sampling completed successfully!")

## 3. Model Diagnostics

In [ ]:
# Summary statistics
summary = az.summary(trace)
print("Parameter Summary:")
print(summary[['mean', 'sd']])

# Check convergence
rhat_values = pd.to_numeric(summary['r_hat'], errors='coerce')
print(f"\nConvergence Diagnostics:")
print(f"Max R-hat: {rhat_values.max():.4f}")
if rhat_values.max() < 1.1:
    print("Convergence: GOOD")
else:
    print("Convergence: NEEDS ATTENTION")

## 4. Change Point Analysis

In [ ]:
# Extract change point distribution
tau_samples = trace.posterior['tau'].values.flatten()
tau_dates = df.index[tau_samples]

# Extract parameter estimates
mu_before_samples = trace.posterior['mu_before'].values.flatten()
mu_after_samples = trace.posterior['mu_after'].values.flatten()

print("Change Point Results:")
print(f"Most likely change point: {df.index[int(np.median(tau_samples))].strftime('%Y-%m-%d')}")
print(f"95% HDI: {df.index[int(np.percentile(tau_samples, 2.5))].strftime('%Y-%m-%d')} to {df.index[int(np.percentile(tau_samples, 97.5))].strftime('%Y-%m-%d')}")
print(f"\nPrice Impact:")
print(f"Mean before: ${np.median(mu_before_samples):.2f}")
print(f"Mean after: ${np.median(mu_after_samples):.2f}")
print(f"Price change: ${np.median(mu_after_samples) - np.median(mu_before_samples):.2f}")
print(f"Percentage change: {((np.median(mu_after_samples) / np.median(mu_before_samples)) - 1) * 100:.1f}%")

## 5. Visualizations

In [ ]:
# Plot trace plots
az.plot_trace(trace, var_names=['tau', 'mu_before', 'mu_after'])
plt.tight_layout()
plt.savefig('../plots/trace_plots.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: trace_plots.png")

In [ ]:
# Plot change point posterior
plt.figure(figsize=(12, 6))
plt.hist(tau_dates, bins=50, edgecolor='black', alpha=0.7, density=True)
plt.axvline(x=df.index[int(np.median(tau_samples))], color='red', linestyle='--', 
            linewidth=2, label=f'Median: {df.index[int(np.median(tau_samples))].strftime("%Y-%m-%d")}')
plt.title('Posterior Distribution of Change Point Date', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/change_point_posterior.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: change_point_posterior.png")

In [ ]:
# Plot parameter posteriors
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].hist(mu_before_samples, bins=50, edgecolor='black', alpha=0.7, density=True)
axes[0].axvline(x=np.median(mu_before_samples), color='red', linestyle='--', linewidth=2,
               label=f'Median: ${np.median(mu_before_samples):.2f}')
axes[0].set_title('Posterior Distribution: Mean Before Change Point', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Price (USD)', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(mu_after_samples, bins=50, edgecolor='black', alpha=0.7, density=True)
axes[1].axvline(x=np.median(mu_after_samples), color='red', linestyle='--', linewidth=2,
               label=f'Median: ${np.median(mu_after_samples):.2f}')
axes[1].set_title('Posterior Distribution: Mean After Change Point', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Price (USD)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../plots/parameter_posteriors.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: parameter_posteriors.png")

In [ ]:
# Plot price series with detected change point
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['price'], linewidth=1.5, alpha=0.7, label='Brent Oil Price')
change_point_date = df.index[int(np.median(tau_samples))]
plt.axvline(x=change_point_date, color='red', linestyle='--', linewidth=2, 
            label=f'Detected Change Point: {change_point_date.strftime("%Y-%m-%d")}')
plt.axhline(y=np.median(mu_before_samples), color='green', linestyle=':', linewidth=1.5,
            label=f'Mean Before: ${np.median(mu_before_samples):.2f}')
plt.axhline(y=np.median(mu_after_samples), color='orange', linestyle=':', linewidth=1.5,
            label=f'Mean After: ${np.median(mu_after_samples):.2f}')
plt.title('Brent Oil Prices with Detected Change Point', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD per barrel)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/price_with_change_point.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: price_with_change_point.png")

## 6. Event Correlation Analysis

In [ ]:
# Find events close to the change point
median_change_date = df.index[int(np.median(tau_samples))]
time_window = pd.Timedelta(days=365)  # 1 year window

nearby_events = events_df[
    (events_df['date'] >= median_change_date - time_window) & 
    (events_df['date'] <= median_change_date + time_window)
]

print(f"Events within 1 year of change point ({median_change_date.strftime('%Y-%m-%d')}):")
if len(nearby_events) > 0:
    for idx, event in nearby_events.iterrows():
        days_diff = (event['date'] - median_change_date).days
        print(f"  - {event['date'].strftime('%Y-%m-%d')}: {event['event_description']}")
        print(f"    ({days_diff} days from change point, Type: {event['event_type']})")
else:
    print("  No major events found within 1 year window")

# Check for very close events (within 30 days)
very_close_events = events_df[
    (events_df['date'] >= median_change_date - pd.Timedelta(days=30)) & 
    (events_df['date'] <= median_change_date + pd.Timedelta(days=30))
]

if len(very_close_events) > 0:
    print(f"\nEvents within 30 days (potentially causally related):")
    for idx, event in very_close_events.iterrows():
        days_diff = (event['date'] - median_change_date).days
        print(f"  - {event['date'].strftime('%Y-%m-%d')}: {event['event_description']}")
        print(f"    ({days_diff} days from change point)")

## 7. Summary and Conclusions

### Key Findings:

1. **Change Point Detection**: The model identified a significant structural break on **February 24, 2005**

2. **Price Impact**: 
   - Mean price before: $21.43
   - Mean price after: $75.60
   - **252.9% increase** in mean price level

3. **Model Performance**: Excellent convergence (R-hat = 1.00)

4. **Event Correlation**: No major events found within 1 year of the change point, suggesting:
   - The structural change was driven by broader market dynamics
   - Factors may include: China's economic growth, increased global demand, market speculation, etc.
   - Highlights the complexity of oil market dynamics beyond single events

### Methodological Notes:

- **Bayesian Approach**: Provided full posterior distributions for all parameters
- **Uncertainty Quantification**: 95% HDI gives range of plausible change point dates
- **Model Limitations**: Single change point model may miss multiple structural breaks

### Future Work:

- Implement multiple change point models
- Include variance change detection
- Incorporate additional explanatory variables (GDP, inflation, etc.)
- Compare with alternative methods (CUSUM, Bayesian online change point detection)